In [2]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [2]:
# Install the necessary libraries (spaCy is critical for NER and segmentation)
!pip install spacy pandas tqdm

# Download the large English language model
!python -m spacy download en_core_web_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 4.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [4]:
import pandas as pd
import spacy
import re
from tqdm import tqdm
import os

# --- Configuration ---
# Input file: The result of the web scraping step
INPUT_SCRAPED_DATA_PATH = '/content/drive/MyDrive/NLP_Project_Scraped_Articles.csv'
# Output file: The preprocessed data ready for fine-tuning/evaluation
OUTPUT_PREPROCESSED_DATA_PATH = '/content/drive/MyDrive/NLP_Project_Preprocessed_Features.csv'

# Load the large English model for robust NER and sentence segmentation
try:
    print("Loading spaCy model 'en_core_web_lg' for NER and Segmentation...")
    # Increase the maximum text length limit for spaCy to handle large documents
    # FIX: We are disabling the 'parser' (which normally handles sentence boundaries),
    # so we must manually add the 'sentencizer' component.
    nlp = spacy.load("en_core_web_lg", disable=["parser", "tagger"])
    nlp.add_pipe('sentencizer') # <-- CRITICAL FIX
    nlp.max_length = 2000000  # Set max length to 2 million characters
except OSError:
    print("ERROR: spaCy model 'en_core_web_lg' not found. Please run the installation step above.")
    nlp = None


def clean_text_for_nlp(text: str) -> str:
    """Performs general text cleaning before spaCy processing."""
    if not isinstance(text, str):
        return ""
    # Remove excessive newlines/tabs and replace with a single space
    text = re.sub(r'\s+', ' ', text).strip()
    # Normalize unicode characters
    # Note: We are using a simple ASCII encode/decode to strip complex non-English characters,
    # which is appropriate since the project focuses on English-language extraction.
    text = text.encode('ascii', 'ignore').decode('ascii')
    return text


def process_article(article_text: str, cameo_code: str, url: str) -> list:
    """
    Applies NLP pipeline (Segmentation, NER, Temporal Tagging) to a single article.

    Returns a list of structured records (one for each sentence/event candidate).
    """
    if not nlp or not article_text:
        return []

    try:
        # Process the article text with spaCy
        doc = nlp(article_text)
    except ValueError as e:
        # Handle case where text length exceeds the (now increased) spaCy limit or other parsing issues
        # print(f"Skipping article due to spaCy processing error: {e}")
        return []

    structured_records = []

    for sent_idx, sentence in enumerate(doc.sents):
        # 1. Tokenization and Segmentation (done by spaCy's nlp(text))
        sent_text = sentence.text.strip()

        if len(sent_text) < 20 or len(sent_text.split()) < 4: # Skip very short or fragmented sentences
             continue

        # 2. Named Entity Recognition (NER)
        # Extract entities and their labels (Person, Organization, Location/Geopolitical Entity)
        entities = []
        for ent in sentence.ents:
            if ent.label_ in ['PERSON', 'ORG', 'GPE', 'LOC']:
                entities.append({
                    'text': ent.text,
                    'label': ent.label_
                })

        # 3. Temporal Tagging (Initial Extraction using simple regex)
        # Finds common written dates (e.g., "August 10, 2025") or year numbers.
        dates = re.findall(r'\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\s+\d{1,2}(?:st|nd|rd|th)?,\s+\d{4}\b|\b\d{1,2}/\d{1,2}/\d{2,4}\b|\b\d{4}\b', sent_text)

        # Normalize the entities and dates into simple strings for the feature list
        entity_features = "|".join([f"{e['label']}:{e['text'].replace('|', '')}" for e in entities])
        temporal_features = "|".join(dates)

        # Append the structured record
        structured_records.append({
            'ArticleURL': url,
            'CAMEO_CODE': cameo_code,
            'SentenceID': f"{url}_{sent_idx}",
            'SentenceText': sent_text,
            'NER_Features': entity_features,
            'Temporal_Features': temporal_features
        })

    return structured_records


def run_advanced_preprocessing():
    """Main function to load scraped data, preprocess, and save features."""
    if not nlp:
        return

    print(f"Loading scraped data from: {INPUT_SCRAPED_DATA_PATH}")
    try:
        df_scraped = pd.read_csv(INPUT_SCRAPED_DATA_PATH)
    except FileNotFoundError:
        print(f"ERROR: Scraped data file not found at {INPUT_SCRAPED_DATA_PATH}. Please ensure scraping completed.")
        return
    except Exception as e:
        print(f"ERROR loading CSV: {e}")
        return

    # Filter 1: Remove articles where scraping failed
    df_clean = df_scraped[~df_scraped['ArticleText'].fillna('').str.startswith('[')].copy()

    print(f"Initial scraped records: {len(df_scraped)}")
    print(f"Viable articles for processing (Text OK): {len(df_clean)}")

    # Apply initial cleaning
    df_clean['CleanText'] = df_clean['ArticleText'].apply(clean_text_for_nlp)

    # Filter 2: Drop rows where cleaning resulted in an empty or very short string
    df_clean = df_clean[df_clean['CleanText'].str.len() > 100].reset_index(drop=True)
    print(f"Articles remaining after cleaning and length check: {len(df_clean)}")

    if len(df_clean) == 0:
        print("No viable articles left after filtering. Exiting.")
        return

    # --- Process all articles and aggregate sentences ---
    all_sentence_records = []

    print(f"Starting segmentation and feature extraction across {len(df_clean)} articles...")

    # Process each article using tqdm to track progress over the iterator
    # We use itertuples() which is faster than iterrows()
    for row in tqdm(df_clean.itertuples(), total=len(df_clean), desc="Processing Articles for Features"):
        # Access columns using attribute notation (e.g., row.CleanText)
        sentence_records = process_article(row.CleanText, row.CAMEO_CODE, row.ArticleURL)
        all_sentence_records.extend(sentence_records)

    df_preprocessed = pd.DataFrame(all_sentence_records)

    if df_preprocessed.empty:
        print("Failed to generate any sentence records. Exiting.")
        return

    # Save the final preprocessed, feature-engineered data
    print(f"\nTotal sentence records generated: {len(df_preprocessed)}")
    df_preprocessed.to_csv(OUTPUT_PREPROCESSED_DATA_PATH, index=False)
    print(f"Preprocessed features saved to: {OUTPUT_PREPROCESSED_DATA_PATH}")


if __name__ == '__main__':
    # We need to import os here for the path checking
    import os
    run_advanced_preprocessing()

Loading spaCy model 'en_core_web_lg' for NER and Segmentation...
Loading scraped data from: /content/drive/MyDrive/NLP_Project_Scraped_Articles.csv
Initial scraped records: 100000
Viable articles for processing (Text OK): 86268
Articles remaining after cleaning and length check: 86268
Starting segmentation and feature extraction across 86268 articles...


Processing Articles for Features:   0%|          | 0/86268 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/spacy/pipeline/lemmatizer.py:188: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger'+'attribute_ruler' or 'morphologizer'.
  warnings.warn(Warnings.W108)
Processing Articles for Features: 100%|██████████| 86268/86268 [3:03:52<00:00,  7.82it/s]



Total sentence records generated: 2805639
Preprocessed features saved to: /content/drive/MyDrive/NLP_Project_Preprocessed_Features.csv
